# Run the experiment on Kaggle (32B model)

Runs our programs on Qwen2.5-Coder 32B exactly as described in `EXPERIMENT.md`. Kaggle's free **GPU T4 ×2** has enough memory for the 32B model (about 20 GB); a free Colab T4 doesn't.

**Before you start**
1. On GitHub, click **Code → Download ZIP** for the `misleading-comments` repo.
2. On Kaggle: **Create → New Notebook**, then **File → Import Notebook** and upload this file.
3. In the right-hand panel under **Session options**: set **Accelerator** to **GPU T4 x2** and turn **Internet** on. (Both need a phone-verified account.)
4. In the right-hand panel: **Add Input → Upload**, drop in the ZIP from step 1, give it any name, and click **Create**.
5. In step 5 below, set `AUTHOR` (or `PROGRAMS`).

**Then run it one of two ways**
- **Try it first:** set `PROGRAMS = ["one_program"]` and run the cells in order. Step 5 prints how long each program took.
- **Full run in the background:** click **Save Version → Save & Run All (Commit) → Save**. Kaggle runs the whole notebook (up to about 12 hours) even if you close the tab. When it finishes, open that version's **Output** and download `results.zip`.

GPU time comes out of Kaggle's weekly free quota, so stop the session (power icon, top right) when you're not using it.

## 1. Check the GPUs
Should list two Tesla T4s.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Copy the repo in
Finds the ZIP you added as an input and copies the repo to `/kaggle/working/repo` (inputs are read-only).

In [ ]:
import glob, os, shutil, sys, zipfile

REPO = "/kaggle/working/repo"
found = glob.glob("/kaggle/input/**/run.py", recursive=True)
if not found:  # Kaggle kept the upload as a ZIP
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if not zips:
        raise RuntimeError("Repo not found: add the GitHub ZIP with Add Input → Upload (see the top of this notebook)")
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/tmp/repo_zip")
    found = glob.glob("/tmp/repo_zip/**/run.py", recursive=True)
shutil.copytree(os.path.dirname(found[0]), REPO, dirs_exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print("Repo ready at", REPO)

## 3. Install and start Ollama

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request

def start_ollama():
    """Start the Ollama server if it isn't already running."""
    for attempt in range(60):
        try:
            urllib.request.urlopen("http://localhost:11434/api/version")
            print("Ollama is running")
            return
        except Exception:
            if attempt == 0:
                # Own session, so interrupting a cell doesn't stop the server.
                subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "a"),
                                 stderr=subprocess.STDOUT, start_new_session=True)
            time.sleep(1)
    raise RuntimeError("Ollama didn't start; see /kaggle/working/ollama.log")

start_ollama()

## 4. Download the model
The exact 32B build for everyone (about 20 GB, roughly 5–10 minutes). Don't change it without the team.

In [ ]:
import run
run.MODEL = "qwen2.5-coder:32b-instruct-q4_K_M"  # all other settings come from run.py
print("Model:", run.MODEL)
!ollama pull {run.MODEL}

## 5. Run
Set `AUTHOR` to your name to run all your programs that don't have 32B results yet. Or list folder names in `PROGRAMS`. Leave both empty to run every program.

In [ ]:
AUTHOR = ""    # your name as in info.json, e.g. "Ankit": runs your programs that don't have 32B results yet
PROGRAMS = []  # or list folder names, e.g. ["chunk_sums"]. Leave both empty to run every program

import json
from pathlib import Path
start_ollama()  # restarts the server if it stopped
!ollama pull {run.MODEL}  # instant if already downloaded; finishes an interrupted download
all_programs = sorted(p for p in os.listdir("programs") if os.path.isdir(os.path.join("programs", p)))
if PROGRAMS:
    names = PROGRAMS
elif AUTHOR:
    names = [p for p in all_programs
             if json.load(open(f"programs/{p}/info.json"))["author"].strip().lower() == AUTHOR.strip().lower()
             and not (run.results_dir() / f"{p}.json").exists()]
else:
    names = all_programs
if not names:
    raise RuntimeError("No programs to run: check the AUTHOR spelling, or they all have results already")

ollama_version, gpu = run.machine_info()
print(f"Model: {run.MODEL}, Ollama {ollama_version}, GPU {gpu}")
print(f"Running {len(names)} programs:", ", ".join(names))
start, failed = time.time(), []
for i, name in enumerate(names, 1):
    t = time.time()
    if not run.run_program(Path("programs") / name):
        failed.append(name)
    per_program = (time.time() - start) / i
    print(f"      {time.time() - t:.0f}s for this program; about {per_program * (len(names) - i) / 60:.0f} min left")
    if i == 1:
        !ollama ps
if failed:
    raise RuntimeError(f"Skipped {', '.join(failed)}: they fail check.py")

`ollama ps` (printed after the first program) should say **100% GPU**. If it shows any CPU share, the model didn't fit on the GPUs and runs very slowly: check that the accelerator is **GPU T4 x2**, not a single T4 or P100.

To rerun with different programs, edit `AUTHOR` or `PROGRAMS` and run this cell again. No need to reinstall or re-upload.

## 6. Summary

In [ ]:
for path in sorted(run.results_dir().glob("*.json")):
    for r in json.load(open(path)):
        print(f'{r["program"]:<26} prompt {r["prompt"]}  {r["version"]:<15} {"PASS" if r["success"] else "FAIL"}')

## 7. Save the results as a ZIP

In [ ]:
folder = os.path.relpath(run.results_dir(), REPO)  # results/qwen2.5-coder-32b
if not os.path.isdir(os.path.join(REPO, folder)):
    raise RuntimeError("No results yet; step 5 didn't finish. Rerun step 5.")
shutil.make_archive("/kaggle/working/results", "zip", REPO, folder)
print("Saved /kaggle/working/results.zip")

Download `results.zip`: in an interactive session, find it under **Output** (`/kaggle/working`) in the right-hand panel and choose **Download**; after a background run, open the version and go to its **Output** tab.

Unzip it, then add its `results/` files to the repo:
- with git: copy them into your local `results/` folder, commit and push, or
- on GitHub: open the repo, **Add file → Upload files**, and drag in the `results` folder.

Don't edit the result files by hand.